# Broiler validation: step-by-step
Set the export directory below, then select **Run All**. This notebook previews each processing stage on one frame, runs the actual pipeline on every exported clip, evaluates the results, and creates a video with ground-truth and estimated boxes. Processing long clips can take time.

The export must contain `test_videos/metadata.csv` (`video_name`), the videos it lists, `ground_truth/events.csv` (`video_name`, optionally `pixel_to_cm`), and `ground_truth/gt_boxes.csv` (`video_name`, `frame_idx`, `x`, `y`, `w`, `h`, `weight_kg`, `event_id`). Boxes are pixel coordinates in `(x, y, width, height)` format. Set the offset to 1 if your ground truth numbers the first frame 0; the pipeline numbers it 1.

In [1]:
from pathlib import Path

# Change this to your FlockLabel export directory (an absolute path also works).
EXPORT_DIR = Path('/Users/akintanoreofeoluwa/Downloads/dataset_2026-11-14_1446').expanduser().resolve()
RESULTS_DIR = EXPORT_DIR / 'results'
GT_FRAME_OFFSET = 0  # 0 for one-based GT frame numbers; 1 for zero-based
IOU_THRESHOLD = 0.3
PREVIEW_VIDEO = None  # None selects the first video; otherwise use its exact filename
PREVIEW_FRAME = 1  # one-based pipeline frame number

## 1. Load and inspect the export
Start Jupyter from the repository root. The tables and first frame below confirm which inputs are being used.

In [2]:
import csv
import sys
import subprocess
import cv2
from IPython.display import display, Image, Video

ROOT = Path.cwd().resolve()
if not (ROOT / 'src' / 'run_eval.py').is_file():
    raise FileNotFoundError('Start Jupyter from the repository root (the folder containing src/).')
sys.path.insert(0, str(ROOT / 'src'))
from FeederCalibration import calibrate_pixel_to_cm
from Factories import AreaCalculatorFactory, AreaMethod, SegmenterFactory, SegmentationMethod
from ObjectFilterStrategy import SimpleObjectFilter
from compare_video import write_comparison

def run_command(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, command)

metadata_path = EXPORT_DIR / 'test_videos' / 'metadata.csv'
events_path = EXPORT_DIR / 'ground_truth' / 'events.csv'
gt_path = EXPORT_DIR / 'ground_truth' / 'gt_boxes.csv'
for path in (metadata_path, events_path, gt_path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing export file: {path}')
with metadata_path.open(newline='') as f:
    videos = list(csv.DictReader(f))
with events_path.open(newline='') as f:
    events = list(csv.DictReader(f))
with gt_path.open(newline='') as f:
    gt_rows = list(csv.DictReader(f))
if not videos or not gt_rows:
    raise ValueError('The export must contain at least one video and one ground-truth box.')
names = [row['video_name'] for row in videos]
missing_videos = [name for name in names if not (EXPORT_DIR / 'test_videos' / name).is_file()]
if missing_videos:
    raise FileNotFoundError(f'Videos listed in metadata but missing from export: {missing_videos}')
unknown_gt = sorted({row['video_name'] for row in gt_rows} - set(names))
if unknown_gt:
    raise ValueError(f'Ground-truth videos not listed in metadata: {unknown_gt}')
print(f'Export: {EXPORT_DIR}\nVideos: {len(videos)} | events: {len(events)} | GT boxes: {len(gt_rows)}')
display(videos[:5], events[:5], gt_rows[:5])
video_name = PREVIEW_VIDEO or names[0]
if video_name not in names:
    raise ValueError(f'Preview video is not in metadata: {video_name}')
video_path = EXPORT_DIR / 'test_videos' / video_name
if PREVIEW_FRAME < 1:
    raise ValueError('PREVIEW_FRAME must be one-based and at least 1')
cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError(f'Cannot open video: {video_path}')
try:
    cap.set(cv2.CAP_PROP_POS_FRAMES, PREVIEW_FRAME - 1)
    ok, frame = cap.read()
finally:
    cap.release()
if not ok:
    raise RuntimeError(f'Cannot read frame {PREVIEW_FRAME} from {video_path}')
def show_image(image):
    ok, encoded = cv2.imencode('.png', image)
    if not ok:
        raise RuntimeError('Could not encode preview image')
    display(Image(data=encoded.tobytes()))
print(f'Input: {video_name}, frame {PREVIEW_FRAME}, {frame.shape[1]} x {frame.shape[0]} pixels')
show_image(frame)

FileNotFoundError: Start Jupyter from the repository root (the folder containing src/).

## 2. Calibrate pixel size
Use `pixel_to_cm` from the event if supplied; otherwise detect the orange feeder in the first frame. The scale converts pixel measurements into centimeters.

In [ ]:
import math
event_by_video = {row['video_name']: row for row in events}
fixed_scale = event_by_video.get(video_name, {}).get('pixel_to_cm', '')
pixel_to_cm = float(fixed_scale) if fixed_scale else calibrate_pixel_to_cm(str(video_path))
if not math.isfinite(pixel_to_cm) or pixel_to_cm <= 0:
    raise ValueError(f'Invalid pixel_to_cm for {video_name}: {pixel_to_cm}')
print(f'{video_name}: {pixel_to_cm:.6f} cm/pixel ({"events.csv" if fixed_scale else "detected feeder"})')

## 3. Segment foreground
The classical segmenter thresholds brightness, suppresses the orange feeder, and cleans the mask. White pixels are foreground candidates.

In [ ]:
segmenter = SegmenterFactory().create(SegmentationMethod.CLASSICAL)
mask = segmenter.segment(frame)
print(f'Foreground pixels: {cv2.countNonZero(mask):,} / {mask.size:,}')
show_image(mask)

## 4. Filter candidates
Contours are checked for size and borders, split/overlap indicators, and template similarity. The cleaned mask is not necessarily the final selection; selected boxes are shown separately.

In [ ]:
obj_filter = SimpleObjectFilter(target_labels=['Broiler'], template_mask=str(ROOT / 'src' / 'broiler_template.png'), min_area=500.0)
selected, clean_mask = obj_filter.should_process(mask)
print(f'Selected candidates: {len(selected)}')
show_image(clean_mask)
preview = frame.copy()
for obj in selected:
    x, y, w, h = obj.crop_region
    cv2.rectangle(preview, (x, y), (x+w, y+h), (0, 0, 255), 2)
show_image(preview)

## 5. Estimate weight
The calibrated calculator models mask columns as circular slices and multiplies their volume by the assumed density of 900 kg/m³. The output field named `volume` is actually estimated weight in kg.

In [ ]:
calculator = AreaCalculatorFactory().create(AreaMethod.CALIBRATED, pixel_to_cm=pixel_to_cm)
preview_estimates = []
for obj in selected:
    obj = calculator.volume_from_mask(obj)
    preview_estimates.append({'box_xywh': obj.crop_region, 'estimated_weight_kg': obj.volume, 'score': obj.confidence_dt + obj.confidence_tpl})
display(preview_estimates)
if not preview_estimates:
    print('No candidate passed the filter in this preview frame; the full run may find birds in other frames.')

## 6. Run every video
This calls the repository runner, using the same methods as the preview. It writes one results CSV and annotated video per clip. Existing files in the results directory are overwritten.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
run_command([sys.executable, '-u', str(ROOT / 'src' / 'run_eval.py'), str(EXPORT_DIR), '--out', str(RESULTS_DIR)])
for name in names:
    path = RESULTS_DIR / f'{Path(name).stem}_results.csv'
    with path.open(newline='') as f:
        rows = list(csv.DictReader(f))
    print(f'{name}: {len(rows)} estimates -> {path}')
    display(rows[:3])

## 7. Compare with ground truth
Boxes are greedily matched one-to-one within the same video and frame by intersection-over-union (IoU). The evaluator prints frame/event weight errors and detection rate. Estimates on frames without labels count as unmatched estimates, not necessarily confirmed false positives. Event actual weights use the median of matched scale readings. Density suggestions are exploratory; fit on separate calibration events and evaluate on held-out events.

In [ ]:
eval_dir = RESULTS_DIR / 'evaluation'
run_command([sys.executable, '-u', str(ROOT / 'src' / 'evaluate_against_gt.py'), str(gt_path), '--results-dir', str(RESULTS_DIR), '--out', str(eval_dir), '--iou', str(IOU_THRESHOLD), '--gt-frame-offset', str(GT_FRAME_OFFSET)])
for name in ('matched.csv', 'per_event.csv'):
    with (eval_dir / name).open(newline='') as f:
        rows = list(csv.DictReader(f))
    print(f'{name}: {len(rows)} rows; first five:')
    display(rows[:5])

## 8. Watch true vs estimated boxes
Green = ground truth and red = pipeline estimate; labels show weights. All boxes are shown, not just IoU matches. Frames without annotations have no green box. The first video with ground truth is displayed; comparison videos for all labeled clips are saved in `results/comparisons`. If your browser cannot play the OpenCV MP4 codec, open the saved file in a local video player.

In [ ]:
comparison_dir = RESULTS_DIR / 'comparisons'
comparisons = []
for name in sorted({row['video_name'] for row in gt_rows}):
    output = comparison_dir / f'{Path(name).stem}_comparison.mp4'
    write_comparison(EXPORT_DIR / 'test_videos' / name, gt_path, RESULTS_DIR / f'{Path(name).stem}_results.csv', output, GT_FRAME_OFFSET, video_name=name)
    comparisons.append(output)
    print(f'Saved: {output}')
print(f'Displaying {comparisons[0].name} (embedding large videos may take time).')
display(Video(str(comparisons[0]), embed=True))